# Results Visualization

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import defaultdict

# Plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Setup
%load_ext autoreload
%autoreload 2

# Data paths
RUN_ID = "20260310_gpt_4_1_x"
RUN_PATH = Path("outputs") / RUN_ID
print(f"Loading data from: {RUN_PATH}\n")

In [ ]:
# Load memory (RL state transitions)
memory_path = RUN_PATH / "checkpoints" / "memory.csv"
df_memory = pd.read_csv(memory_path)
print(f"Loaded {len(df_memory)} transitions from memory.csv")
print(f"Columns: {df_memory.columns.tolist()}")
print(f"\nShape: {df_memory.shape}")
print(f"\nFirst few rows:")
df_memory.head()

In [ ]:
# Load interaction files to understand structure
interactions_dir = RUN_PATH / "interactions"
interaction_files = sorted(interactions_dir.glob("problem_*.json"))
print(f"Found {len(interaction_files)} interaction files")

# Load one example to explore structure
with open(interaction_files[0]) as f:
    sample_interaction = json.load(f)

print(f"\nSample interaction (problem 0) has {len(sample_interaction)} entries")
print("Structure:")
for i, entry in enumerate(sample_interaction[:3]):
    print(f"  Entry {i}: {list(entry.keys())}")

# Graph 3: Tutor and Student Abstraction Level Distributions

In [ ]:
def plot_abstraction_level_distribution(df, level_col, title, color_rgb, entity_name):
    """
    Create a bar chart for abstraction level distribution.
    
    Parameters:
    - df: DataFrame
    - level_col: column name (e.g., 'tutor_level', 'student_level')
    - title: chart title
    - color_rgb: color string (e.g., 'rgba(255, 182, 193, 0.8)')
    - entity_name: name of entity (e.g., 'Tutor Level', 'Student Level')
    """
    level_counts = df[level_col].value_counts().sort_index()
    
    fig = go.Figure(data=[
        go.Bar(
            x=level_counts.index.astype(str),
            y=level_counts.values,
            marker=dict(color=color_rgb, line=dict(color='darkred', width=2)),
            text=level_counts.values,
            textposition='auto',
            hovertemplate='<b>Level %{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title=title,
        xaxis_title=f"{entity_name} (1=Concrete, 4=Abstract)",
        yaxis_title="Frequency",
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=700,
        font=dict(size=12),
        showlegend=False
    )
    
    fig.show()
    
    print(f"\n{entity_name} Statistics:")
    print(f"Mean: {df[level_col].mean():.2f}")
    print(f"Median: {df[level_col].median():.2f}")
    print(f"Std Dev: {df[level_col].std():.2f}")
    print(f"\nCounts:\n{level_counts}")

# Plot tutor level distribution
plot_abstraction_level_distribution(
    df_memory,
    'tutor_level',
    'Judged Tutor Abstraction Levels',
    'rgba(255, 182, 193, 0.8)',
    'Tutor Level'
)

In [ ]:
# Plot student level distribution
plot_abstraction_level_distribution(
    df_memory,
    'student_level',
    'Judged Student Abstraction Levels',
    'rgba(135, 206, 250, 0.8)',
    'Student Level'
)

# Graph 10: Pedagogical Move Distribution

In [ ]:
# Get pedagogical move distribution
action_counts = df_memory['action'].value_counts()
action_order = ["SOCRATIC_PROBE", "CONCEPTUAL_HINT", "STRUCTURAL_SCAFFOLD"]
action_counts = action_counts.reindex([a for a in action_order if a in action_counts.index])

# Color mapping for actions
action_colors = {
    "SOCRATIC_PROBE": "rgba(100, 149, 237, 0.8)",        # Cornflower blue
    "CONCEPTUAL_HINT": "rgba(144, 238, 144, 0.8)",       # Light green
    "STRUCTURAL_SCAFFOLD": "rgba(255, 165, 0, 0.8)",     # Orange
}

fig = go.Figure(data=[
    go.Bar(
        x=action_counts.index,
        y=action_counts.values,
        marker=dict(
            color=[action_colors.get(action, 'gray') for action in action_counts.index],
            line=dict(color='black', width=1.5)
        ),
        text=action_counts.values,
        textposition='auto',
        hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
    )
])

fig.update_layout(
    title="Distribution of Pedagogical Actions (Training Cycle)",
    xaxis_title="Pedagogical Move",
    yaxis_title="Frequency",
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    height=500,
    width=800,
    font=dict(size=12),
    showlegend=False,
    xaxis=dict(tickangle=-15)
)

fig.show()

print(f"\nPedagogical Move Statistics:")
print(f"Total actions: {action_counts.sum()}")
print(f"\nCounts:")
for action, count in action_counts.items():
    pct = (count / action_counts.sum()) * 100
    print(f"  {action}: {count} ({pct:.1f}%)")

# Graph 10.2: Action with Highest Predicted Reward

In [ ]:
# Extract best actions from predicted rewards
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df_memory[pred_columns].dropna()

if len(df_with_preds) > 0:
    # Find which action had the highest predicted reward for each row
    best_actions = df_with_preds.idxmax(axis=1).str.replace('pred_reward_', '')
    best_action_counts = best_actions.value_counts()
    best_action_counts = best_action_counts.reindex([a for a in action_order if a in best_action_counts.index])
    
    # Create visualization
    fig = go.Figure(data=[
        go.Bar(
            x=best_action_counts.index,
            y=best_action_counts.values,
            marker=dict(
                color=[action_colors.get(action, 'gray') for action in best_action_counts.index],
                line=dict(color='black', width=1.5)
            ),
            text=best_action_counts.values,
            textposition='auto',
            hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title="Distribution of Optimal Actions (Exploitation Cycle)",
        xaxis_title="Pedagogical Move",
        yaxis_title="Frequency",
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=800,
        font=dict(size=12),
        showlegend=False,
        xaxis=dict(tickangle=-15)
    )
    
    fig.show()
    
    # Print statistics
    print(f"\nHighest Predicted Reward Action Statistics:")
    print(f"Rows with predictions: {len(df_with_preds)}")
    print(f"\nCounts:")
    for action, count in best_action_counts.items():
        pct = (count / best_action_counts.sum()) * 100
        print(f"  {action}: {count} ({pct:.1f}%)")
else:
    print("No rows with predicted rewards found")

# Graph 6: Highest Predicted Action Distribution Over Training Time

In [ ]:
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
group_size = 20  # Group every N problems

# Extract best actions from predictions
df_with_preds = df_memory[pred_columns].dropna()

if len(df_with_preds) > 0:
    best_actions = df_with_preds.idxmax(axis=1).str.replace('pred_reward_', '')
    problem_nums = df_memory.loc[best_actions.index, 'problem'].values
    
    # Group by training progress
    df_grouped = pd.DataFrame({
        'best_action': best_actions.values,
        'problem': problem_nums
    })
    df_grouped['group'] = (df_grouped['problem'] // group_size).astype(int)
    
    group_action_counts = df_grouped.groupby(['group', 'best_action']).size().unstack(fill_value=0)
    group_action_counts = group_action_counts.reindex([a for a in action_order if a in group_action_counts.columns], axis=1, fill_value=0)
    
    # Create visualization
    group_labels = [f"Problems {i*group_size}-{(i+1)*group_size-1}" for i in group_action_counts.index]
    
    fig = go.Figure()
    for action in group_action_counts.columns:
        fig.add_trace(go.Scatter(
            x=group_labels,
            y=group_action_counts[action],
            mode='lines',
            name=action,
            line=dict(width=0),
            fillcolor=action_colors.get(action, 'gray'),
            stackgroup='one',
            hovertemplate='<b>%{fullData.name}</b><br>%{x}<br>Count: %{y}<extra></extra>'
        ))
    
    fig.update_layout(
        title=f"Distribution of Optimal Actions Over Training (Grouped by {group_size} Problems)",
        xaxis_title="Training Progress",
        yaxis_title="Frequency",
        hovermode='x unified',
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=900,
        font=dict(size=11),
    )
    
    fig.show()
    
    # Print summary
    print(f"\nHighest Predicted Action Over Training Time (group size={group_size}):")
    print(f"Total groups: {len(group_action_counts)}")
    print(f"\n{group_action_counts}")
else:
    print("No rows with predicted rewards found")